In [ ]:
# **Author: Shakir Showkat Sofi**
!pip install git+https://github.com/PGelss/scikit_tt.git

In [2]:
import numpy as np
import scipy as sp
from scikit_tt import tensor_train as tt

## Utility functions

In [3]:
def block2right(X, n, tol=1e-8):
    """
    Shift block index from core n to core n+1 in a Block-TT-Matrix.
    - X.cores[n]   : (rL, In, K, rR)      # block core
    - X.cores[n+1] : (rR, Inext, 1, rNext)

    Output:
      - core[n]     : (rL, In, 1, rRnew)
      - core[n+1]   : (rRnew, Inext, K, rNext) # block core
    """
    cores = list(X.cores)
    coren = cores[n]
    corenp1 = cores[n+1]

    rL, In, K, rR = coren.shape
    rR_chk, Inext, Jnext, rNext = corenp1.shape
    assert rR == rR_chk, "Rank mismatch between core n and n+1"
    assert Jnext == 1, "Expected singleton physical index in non-block core"

    # reshape (rL*In) x (K*rR)
    M = coren.reshape(rL * In, K * rR)
    Q, R = sp.linalg.qr(M, mode='economic', overwrite_a=True)
    rRnew = Q.shape[1]  # use tol for truncation

    # new ordinary TT-matrix core at n: (rL, In, 1, rRnew)
    coren_new = Q.reshape(rL, In, 1, rRnew)

    # new block core at n+1
    # R: (rRnew, K, rR), corenp1: (rR, Inext, 1, rNext)
    R_sh = R.reshape(rRnew, K, rR)
    corenp1_new = np.einsum('akb,bid->aikd', R_sh, corenp1[:,:,0,:], optimize=True)
    # result: (rRnew, Inext, K, rNext)

    # rebuild core tuple
    newcores = cores[:n] + [coren_new] + [corenp1_new] + cores[n+2:]
    return tt.TT(newcores)

def block2left(X, n, tol=1e-8):
    cores = list(X.cores)
    coren = cores[n]
    coren_prev = cores[n-1]

    rL, In, K, rR = coren.shape
    rPrev, Iprev, Jprev, rL_chk = coren_prev.shape
    assert rL == rL_chk, "Rank mismatch between core n-1 and n"
    assert Jprev == 1, "Expected singleton physical index in non-block core"

    # reshape M = (rL*K) x (In*rR)
    M = coren.transpose(0, 2, 1, 3).reshape(rL*K, In*rR)
    R, Q = sp.linalg.rq(M, mode='economic', overwrite_a=True,)
    rLnew = R.shape[1] # use tol for truncation

    # new block core at n-1: (rPrev, Iprev, K, rLnew)
    R_sh = R.reshape(rL, K, rLnew)
    coren_prev_new = np.einsum('pil,lka->pika', coren_prev[:,:,0,:], R_sh, optimize=True)

    # new ordinary core at n: (rLnew, In, 1, rR)
    coren_new = Q.reshape(rLnew, In, rR)
    coren_new = coren_new.reshape(rLnew, In, 1, rR)

    newcores = cores[:n-1] + [coren_prev_new] + [coren_new] + cores[n+1:]
    return tt.TT(newcores)

In [4]:
# Right interface products
def _right_interface(i: int, stack_R: list[np.ndarray], operator: tt.TT, solution: tt.TT):
    if i == operator.order - 1:
        # last stack element is 1
        stack_R[i] = np.array([1], ndmin=3)
    else:
        # contract previous stack element with solution and operator cores
        stack_R[i] = np.tensordot(np.conj(solution.cores[i + 1][:, :, 0, :]), stack_R[i + 1], axes=(2, 2))
        stack_R[i] = np.tensordot(operator.cores[i + 1], stack_R[i], axes=([1, 3], [1, 3]))
        stack_R[i] = np.tensordot(solution.cores[i + 1][:, :, 0, :], stack_R[i], axes=([1, 2], [1, 3]))

# Left interface products
def _left_interface(i: int, stack_L: list[np.ndarray], operator: tt.TT, solution: tt.TT):
    if i == 0:
        # first stack element is 1
        stack_L[i] = np.array([1], ndmin=3)
    else:
        # contract previous stack element with solution and operator cores
        stack_L[i] = np.tensordot(stack_L[i - 1], solution.cores[i - 1][:, :, 0, :], axes=(0, 0))
        stack_L[i] = np.tensordot(stack_L[i], operator.cores[i - 1], axes=([0, 2], [0, 2]))
        stack_L[i] = np.tensordot(stack_L[i], np.conj(solution.cores[i - 1][:, :, 0, :]), axes=([0, 2], [0, 1]))

# Effective operative: eff_op(mu) = contract(stack_L<mu, Op, stack_R>mu)
def _effective_op(i: int,        stack_L:  list[np.ndarray],
                                 stack_R: list[np.ndarray],
                                 operator: tt.TT, solution: tt.TT) -> np.ndarray:
   # contract stack elements and operator core
    eff_op = np.tensordot(stack_L[i], operator.cores[i], axes=(1, 0))
    eff_op = np.tensordot(eff_op, stack_R[i], axes=(4, 1))

    # transpose and reshape micro matrix
    eff_op = eff_op.transpose([1, 2, 5, 0, 3, 4]).reshape(
        solution.ranks[i] * operator.row_dims[i] * solution.ranks[i + 1],
        solution.ranks[i] * operator.col_dims[i] * solution.ranks[i + 1])

    return eff_op

## SVD/EVD using DMRG

In [5]:
def hilbert_matrix(N): # test matrix
    m = 2**N
    i = np.arange(1, m+1).reshape(-1, 1)   # column vector
    j = np.arange(1, m+1).reshape(1, -1)   # row vector
    return 1.0 / (i + j - 1)

In [6]:
N = 8
D = [2]*N
H = hilbert_matrix(N)
Ht = H.reshape(D+D)
H.shape, Ht.shape

((256, 256), (2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2))

In [7]:
operator = tt.TT(Ht, threshold=1e-5)
operator


Tensor train with order    = 8, 
                  row_dims = [2, 2, 2, 2, 2, 2, 2, 2], 
                  col_dims = [2, 2, 2, 2, 2, 2, 2, 2], 
                  ranks    = [1, 3, 5, 5, 5, 5, 5, 3, 1]

In [8]:
K = 4;
# computing matrix svd
Opmat = operator.matricize()
[uo,so,vo] = np.linalg.svd(Opmat)
so[:K]

array([2.303809  , 1.00376266, 0.32445051, 0.09115996])

DMRG SVD/EVD

In [9]:
# Initialize  TT SVD/EVD
colshp = [1]*N
colshp[0] = K
ttrank = tuple(x for x in operator.ranks)
Uinit = tt.rand(operator.row_dims, colshp, ranks=list(ttrank)).ortho_right() # first core unnormalized

For SVD, use `U` and `V`. Here, we only use `U`, which is EVD case.

In [22]:
# ALS
orderswp = list(range(N-1)) + list(range(N-1, 0, -1))
U = Uinit.copy()
Op = operator.copy()
order = U.order
nswps = 5
swp = 1
while swp<=nswps:
    incr=True
    swp+=1
    for n in orderswp:
        stack_L = [None] * order
        stack_R = [None] * order
        stack_L[0] = np.ones((1, 1, 1))
        stack_R[-1] = np.ones((1, 1, 1))

        if n==N-1:
            incr=False
        # --- Build left stack ---
        for i in range(n+1):
            _left_interface(i, stack_L, Op, U)

        # --- Build right stack ---
        for i in reversed(range(n, N)):
            _right_interface(i, stack_R, Op, U)

        # --- Construct effective operator for core n ---
        eff_op = _effective_op(n, stack_L, stack_R, Op, U)

        Uop, Sop, Vop = np.linalg.svd(eff_op, full_matrices=True)
        K2 = np.minimum(K, Uop.shape[1])
        Ss = Sop[:K2]
        Us = Uop[:, :K2]
        Vs = Vop[:K2, :]

        if incr:
            U.cores[n] = Us.reshape(U.ranks[n], U.row_dims[n], U.ranks[n+1], K2).transpose(0, 1, 3, 2)
            U = block2right(U, n)
        else:
            U.cores[n] = Vs.reshape(K2, U.ranks[n], U.row_dims[n], U.ranks[n+1]).transpose(1, 2, 0, 3)
            U = block2left(U, n)

    sse = sp.linalg.subspace_angles(U.matricize(), uo[:,:K])
    print('Subspace Error\n', sse)

    mse = np.linalg.norm(U.matricize()@np.diag(Ss)@U.matricize().conj().T-Opmat)
    print('Norm Error\n', mse)

Subspace Error
 [9.96233817e-15 4.99380500e-15 3.27348601e-15 2.01562427e-15]
Norm Error
 0.024382273144967555
Subspace Error
 [1.03303362e-14 4.91758675e-15 3.54343416e-15 2.35818520e-15]
Norm Error
 0.024382273144967517
Subspace Error
 [9.48348527e-15 5.05734272e-15 3.53433843e-15 1.96297807e-15]
Norm Error
 0.024382273144967537
Subspace Error
 [1.03819166e-14 4.75412271e-15 3.18018007e-15 1.94996074e-15]
Norm Error
 0.024382273144967534
Subspace Error
 [9.20395669e-15 4.98982607e-15 3.34764666e-15 1.85962650e-15]
Norm Error
 0.02438227314496751


In [23]:
so[:K]

array([2.303809  , 1.00376266, 0.32445051, 0.09115996])

In [24]:
Ss

array([2.303809  , 1.00376266, 0.32445051, 0.09115996])

In [25]:
U


Tensor train with order    = 8, 
                  row_dims = [2, 2, 2, 2, 2, 2, 2, 2], 
                  col_dims = [4, 1, 1, 1, 1, 1, 1, 1], 
                  ranks    = [1, 8, 16, 32, 16, 8, 4, 2, 1]

In [26]:
uo[:,:K]

array([[-0.48306577,  0.61147862,  0.48643   ,  0.32062866],
       [-0.32867374,  0.25618178, -0.06724271, -0.34158428],
       [-0.26381602,  0.13014245, -0.1865331 , -0.33076787],
       ...,
       [-0.01418164, -0.02716129,  0.0378999 , -0.04691382],
       [-0.01413484, -0.02708565,  0.03783601, -0.0469207 ],
       [-0.01408831, -0.02701029,  0.03777187, -0.04692621]])

In [27]:
U.matricize()

array([[-0.48306577,  0.61147862, -0.48643   ,  0.32062866],
       [-0.32867374,  0.25618178,  0.06724271, -0.34158428],
       [-0.26381602,  0.13014245,  0.1865331 , -0.33076787],
       ...,
       [-0.01418164, -0.02716129, -0.0378999 , -0.04691382],
       [-0.01413484, -0.02708565, -0.03783601, -0.0469207 ],
       [-0.01408831, -0.02701029, -0.03777187, -0.04692621]])